<a href="https://colab.research.google.com/github/nisabulut/digital-image-processing/blob/main/10_1_goruntu_isleme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
%%bash
cat << 'EOF' > stereo_bm.cpp
#include <opencv2/opencv.hpp>
#include <iostream>

int main() {
    cv::Mat imgLeft  = cv::imread("scene1.row3.col1.ppm", cv::IMREAD_GRAYSCALE);
    cv::Mat imgRight = cv::imread("scene1.row3.col3.ppm", cv::IMREAD_GRAYSCALE);

    if (imgLeft.empty() || imgRight.empty()) {
        std::cerr << "Goruntuler yuklenemedi!" << std::endl;
        return -1;
    }

    int numDisparities = 16 * 8; // 16'nın katı olmalı
    int blockSize = 15;      // 5..255 arası tek sayı (BM için genelde 11-21 arası kullanılır)

    cv::Ptr<cv::StereoBM> bm = cv::StereoBM::create(numDisparities, blockSize);

    // İsteğe bağlı ince ayar parametreleri
    bm->setMinDisparity(0);
    bm->setTextureThreshold(10);
    bm->setUniquenessRatio(15);
    bm->setSpeckleWindowSize(100);
    bm->setSpeckleRange(32);
    bm->setDisp12MaxDiff(1);

    cv::Mat disp16S, dispFloat, dispNormalized, dispColor;
    bm->compute(imgLeft, imgRight, disp16S);

    // 16'ya bölerek gerçek piksel disparite değerine çevir
    disp16S.convertTo(dispFloat, CV_32F, 1.0 / 16.0);

    // Görselleştirme için 0-255 aralığına normalize et ve renklendir
    double minVal, maxVal;
    cv::minMaxLoc(dispFloat, &minVal, &maxVal);
    double scale = (maxVal - minVal > 0) ? (255.0 / (maxVal - minVal)) : 1.0;
    dispFloat.convertTo(dispNormalized, CV_8U, scale, -minVal * scale);

    cv::applyColorMap(dispNormalized, dispColor, cv::COLORMAP_TURBO);
    cv::imwrite("disparity_bm.png", dispColor);

    std::cout << "Tamamlandi: disparity_bm.png olusturuldu." << std::endl;
    return 0;
}
EOF

g++ stereo_bm.cpp -o stereo_bm `pkg-config --cflags --libs opencv4`
./stereo_bm

Tamamlandi: disparity_bm.png olusturuldu.
